# Manual Test Plan - Shell-Based Notebook

This notebook executes the manual test plan using bash cells.

## Usage
- Run the **Setup** cell first to configure environment
- Run individual test sections as needed (each cell is a bash cell)
- Use the **Report** cell to generate a summary
- Cells with `SKIP` are marked for sections requiring auth or known limitations

## 0. Setup - Environment Configuration

In [1]:
%%bash

# Environment configuration
export PR_TEST_ROOT=$HOME/.test-prompt-registry
export XDG_CONFIG_HOME="$PR_TEST_ROOT/xdg"
export REPO_ROOT=$(cd /home/wherka/workspace/opensource/prompt-registry && pwd)
export PR_BIN="node $REPO_ROOT/lib/dist/cli/main.js"

# Create test directories
rm -rf "$PR_TEST_ROOT"
mkdir -p "$PR_TEST_ROOT"/{xdg,project,bundles/local-foo,exports}

# Create results tracking file
echo '{"sections":{}}' > "$PR_TEST_ROOT/results.json"

echo "Test root: $PR_TEST_ROOT"
echo "CLI binary: $PR_BIN"
echo "XDG config: $XDG_CONFIG_HOME"
echo ""
echo "Setup complete. Ready to run tests."

Test root: /home/wherka/.test-prompt-registry
CLI binary: node /home/wherka/workspace/opensource/prompt-registry/lib/dist/cli/main.js
XDG config: /home/wherka/.test-prompt-registry/xdg

Setup complete. Ready to run tests.


## Helper Functions (bash)

In [2]:
%%bash

# Helper function to record test results
record_result() {
    local section="$1"
    local test_name="$2"
    local status="$3"  # pass, fail, skip
    local output="$4"
    local duration="$5"
    
    local results_file="$PR_TEST_ROOT/results.json"
    
    # Use jq to update results
    jq --arg s "$section" \
       --arg n "$test_name" \
       --arg st "$status" \
       --arg o "$output" \
       --arg d "$duration" \
       '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
       "$results_file" > "$results_file.tmp" && mv "$results_file.tmp" "$results_file"
}

echo "Helper functions loaded."

Helper functions loaded.


## 1. Unauthenticated Smoke Test

In [3]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
export XDG_CONFIG_HOME="$PR_TEST_ROOT/xdg"
export REPO_ROOT=$(cd /home/wherka/workspace/opensource/prompt-registry && pwd)
export PR_BIN="node $REPO_ROOT/lib/dist/cli/main.js"
EOF

# 1.1 Version check
start=$(date +%s.%N)
output=$($PR_BIN --version 2>&1)
end=$(date +%s.%N)
duration=$(echo "$end - $start" | bc)

if [ $? -eq 0 ] && [ -n "$output" ]; then
    status="pass"
else
    status="fail"
fi

jq --arg s "1" --arg n "version" --arg st "$status" --arg o "$output" --arg d "$duration" \
   '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
   "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"

echo "Version check: $status"
echo "$output"

Version check: pass
1.0.4


In [4]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
export XDG_CONFIG_HOME="$PR_TEST_ROOT/xdg"
export REPO_ROOT=$(cd /home/wherka/workspace/opensource/prompt-registry && pwd)
export PR_BIN="node $REPO_ROOT/lib/dist/cli/main.js"
EOF

# 1.2 Help check
start=$(date +%s.%N)
output=$($PR_BIN --help 2>&1)
end=$(date +%s.%N)
duration=$(echo "$end - $start" | bc)

if [ $? -eq 0 ] && echo "$output" | grep -q "prompt-registry"; then
    status="pass"
else
    status="fail"
fi

jq --arg s "1" --arg n "help" --arg st "$status" --arg o "$(echo "$output" | head -5)" --arg d "$duration" \
   '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
   "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"

echo "Help check: $status"

Help check: pass


In [5]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
export XDG_CONFIG_HOME="$PR_TEST_ROOT/xdg"
export REPO_ROOT=$(cd /home/wherka/workspace/opensource/prompt-registry && pwd)
export PR_BIN="node $REPO_ROOT/lib/dist/cli/main.js"
EOF

# 1.3 Empty-state list
unset GITHUB_TOKEN GH_TOKEN
export PROMPT_REGISTRY_DISABLE_GH_CLI=1

start=$(date +%s.%N)
output=$($PR_BIN hub list -o json 2>&1)
end=$(date +%s.%N)
duration=$(echo "$end - $start" | bc)

if [ $? -eq 0 ] && echo "$output" | jq -e '.status=="ok" and .data.hubs==[] and .data.activeId==null' > /dev/null; then
    status="pass"
else
    status="fail"
fi

jq --arg s "1" --arg n "empty-state-list" --arg st "$status" --arg o "$output" --arg d "$duration" \
   '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
   "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"

echo "Empty state list: $status"
echo "$output"

unset PROMPT_REGISTRY_DISABLE_GH_CLI

Empty state list: pass
{"schemaVersion":1,"command":"hub.list","status":"ok","data":{"hubs":[],"activeId":null},"warnings":[],"errors":[],"meta":{}}


In [6]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
export XDG_CONFIG_HOME="$PR_TEST_ROOT/xdg"
export REPO_ROOT=$(cd /home/wherka/workspace/opensource/prompt-registry && pwd)
export PR_BIN="node $REPO_ROOT/lib/dist/cli/main.js"
EOF

# 1.4 Unknown command
start=$(date +%s.%N)
$PR_BIN bogus 2>&1
exit_code=$?
end=$(date +%s.%N)
duration=$(echo "$end - $start" | bc)

if [ $exit_code -ne 0 ]; then
    status="pass"
else
    status="fail"
fi

jq --arg s "1" --arg n "unknown-command" --arg st "$status" --arg o "exit code: $exit_code" --arg d "$duration" \
   '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
   "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"

echo "Unknown command: $status (exit code: $exit_code)"

Command not found; did you mean one of:

  0. prompt-registry -h
  1. prompt-registry -v
  2. prompt-registry bundle build
  3. prompt-registry bundle manifest
  4. prompt-registry collection affected
  5. prompt-registry collection list
  6. prompt-registry collection validate
  7. prompt-registry doctor
  8. prompt-registry explain
  9. prompt-registry index search
 10. prompt-registry index stats
 11. prompt-registry index build
 12. prompt-registry index shortlist new
 13. prompt-registry index export
 14. prompt-registry index eval
 15. prompt-registry index bench
 16. prompt-registry index harvest
 17. prompt-registry index report
 18. prompt-registry skill new
 19. prompt-registry skill validate
 20. prompt-registry version compute
 21. prompt-registry config list
 22. prompt-registry plugins list
 23. prompt-registry target list
 24. prompt-registry install
 25. prompt-registry uninstall
 26. prompt-registry config get
 27. prompt-registry target add
 28. prompt-registry target

In [7]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
export XDG_CONFIG_HOME="$PR_TEST_ROOT/xdg"
export REPO_ROOT=$(cd /home/wherka/workspace/opensource/prompt-registry && pwd)
export PR_BIN="node $REPO_ROOT/lib/dist/cli/main.js"
EOF

# 1.5 Index unknown verb
start=$(date +%s.%N)
$PR_BIN index ghost 2>&1
exit_code=$?
end=$(date +%s.%N)
duration=$(echo "$end - $start" | bc)

if [ $exit_code -eq 64 ]; then
    status="pass"
else
    status="fail"
fi

jq --arg s "1" --arg n "index-unknown-verb" --arg st "$status" --arg o "exit code: $exit_code" --arg d "$duration" \
   '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
   "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"

echo "Index unknown verb: $status (exit code: $exit_code)"

Command not found; did you mean one of:

  0. prompt-registry index search
  1. prompt-registry index stats
  2. prompt-registry index build
  3. prompt-registry index shortlist new
  4. prompt-registry index export
  5. prompt-registry index eval
  6. prompt-registry index bench
  7. prompt-registry index harvest
  8. prompt-registry index report

While running index ghost
Index unknown verb: pass (exit code: 64)


In [8]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
export XDG_CONFIG_HOME="$PR_TEST_ROOT/xdg"
export REPO_ROOT=$(cd /home/wherka/workspace/opensource/prompt-registry && pwd)
export PR_BIN="node $REPO_ROOT/lib/dist/cli/main.js"
EOF

# 1.6 Explain produces output
start=$(date +%s.%N)
output=$($PR_BIN explain INDEX.NOT_FOUND 2>&1)
line_count=$(echo "$output" | wc -l)
end=$(date +%s.%N)
duration=$(echo "$end - $start" | bc)

if [ $line_count -gt 0 ]; then
    status="pass"
else
    status="fail"
fi

jq --arg s "1" --arg n "explain-output" --arg st "$status" --arg o "lines: $line_count" --arg d "$duration" \
   '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
   "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"

echo "Explain output: $status ($line_count lines)"

Explain output: pass (3 lines)


In [9]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
export XDG_CONFIG_HOME="$PR_TEST_ROOT/xdg"
export REPO_ROOT=$(cd /home/wherka/workspace/opensource/prompt-registry && pwd)
export PR_BIN="node $REPO_ROOT/lib/dist/cli/main.js"
EOF

# 1.7 Output format matrix
all_passed=true
for fmt in text json yaml ndjson; do
    start=$(date +%s.%N)
    output=$($PR_BIN hub list -o $fmt 2>&1)
    end=$(date +%s.%N)
    duration=$(echo "$end - $start" | bc)
    
    if [ $? -eq 0 ]; then
        if [ "$fmt" != "ndjson" ] && [ -z "$output" ]; then
            status="fail"
            all_passed=false
        else
            status="pass"
        fi
    else
        status="fail"
        all_passed=false
    fi
    
    jq --arg s "1" --arg n "output-format-$fmt" --arg st "$status" --arg o "" --arg d "$duration" \
       '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
       "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"
    
    echo "  -o $fmt: $status"
done

echo ""
if [ "$all_passed" = true ]; then
    echo "All output formats: pass"
else
    echo "All output formats: fail"
fi

  -o text: pass
  -o json: pass
  -o yaml: pass
  -o ndjson: pass

All output formats: pass


## 2. Initiate User Configuration from Real Hub (SKIP - requires GitHub auth)

In [ ]:
%%bash

# Environment configuration
export PR_TEST_ROOT=$HOME/.test-prompt-registry
export XDG_CONFIG_HOME="$PR_TEST_ROOT/xdg"
export REPO_ROOT=$(cd /home/wherka/workspace/opensource/prompt-registry && pwd)
export PR_BIN="node $REPO_ROOT/lib/dist/cli/main.js"

# Create test directories
rm -rf "$PR_TEST_ROOT"
mkdir -p "$PR_TEST_ROOT"/{xdg,project,bundles/local-foo,exports}

# Write environment to a file for sourcing in other cells
cat > "$PR_TEST_ROOT/.env" <<ENV
export PR_TEST_ROOT="$PR_TEST_ROOT"
export XDG_CONFIG_HOME="$XDG_CONFIG_HOME"
export REPO_ROOT="$REPO_ROOT"
export PR_BIN="$PR_BIN"
ENV

# Create results tracking file
echo '{"sections":{}}' > "$PR_TEST_ROOT/results.json"

echo "Test root: $PR_TEST_ROOT"
echo "CLI binary: $PR_BIN"
echo "XDG config: $XDG_CONFIG_HOME"
echo ""
echo "Setup complete. Ready to run tests."
echo ""
echo "In subsequent cells, use: source \"$PR_TEST_ROOT/.env\""

source "$PR_TEST_ROOT/.env"

gh auth status

$PR_BIN hub add \
  --type github \
  --location Amadeus-xDLC/genai.prompt-registry-config \
  --ref main \
  -o json

jq --arg s "2" --arg n "skip-auth-required" --arg st "skip" --arg o "Skipped - requires GitHub auth" --arg d "0" \
   '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
   "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"

echo "Section 2 skipped - requires GitHub authentication"

Test root: /home/wherka/.test-prompt-registry
CLI binary: node /home/wherka/workspace/opensource/prompt-registry/lib/dist/cli/main.js
XDG config: /home/wherka/.test-prompt-registry/xdg

Setup complete. Ready to run tests.

In subsequent cells, use: source "/home/wherka/.test-prompt-registry/.env"


hub-config.yml not found at Amadeus-xDLC/genai.prompt-registry-config


Section 2 skipped - requires GitHub authentication


## 3. Browse Profiles from Hub (SKIP - depends on section 2)

In [ ]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
EOF

jq --arg s "3" --arg n "skip-depends-on-section-2" --arg st "skip" --arg o "Skipped - depends on section 2" --arg d "0" \
   '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
   "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"

echo "Section 3 skipped - depends on section 2"

## 4. Project-level Configuration (SKIP - target add is a defineCommand)

In [ ]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
EOF

jq --arg s "4" --arg n "skip-definecommand-limitation" --arg st "skip" --arg o "Skipped - target add is a defineCommand (known limitation)" --arg d "0" \
   '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
   "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"

echo "Section 4 skipped - target add is a defineCommand (known limitation)"

## 5. Detached / Default-local Hub Flow

In [ ]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
export XDG_CONFIG_HOME="$PR_TEST_ROOT/xdg"
export REPO_ROOT=$(cd /home/wherka/workspace/opensource/prompt-registry && pwd)
export PR_BIN="node $REPO_ROOT/lib/dist/cli/main.js"
EOF

# 5.1 Add detached source
cd "$PR_TEST_ROOT/project"
start=$(date +%s.%N)
output=$($PR_BIN source add --type github --url owner/repo --id detached-foo -o json 2>&1)
end=$(date +%s.%N)
duration=$(echo "$end - $start" | bc)

if echo "$output" | jq -e '.status=="ok"' > /dev/null; then
    status="pass"
else
    status="fail"
fi

jq --arg s "5" --arg n "source-add" --arg st "$status" --arg o "$output" --arg d "$duration" \
   '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
   "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"

echo "Source add: $status"
echo "$output"

In [ ]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
export XDG_CONFIG_HOME="$PR_TEST_ROOT/xdg"
export REPO_ROOT=$(cd /home/wherka/workspace/opensource/prompt-registry && pwd)
export PR_BIN="node $REPO_ROOT/lib/dist/cli/main.js"
EOF

# 5.2 List sources
cd "$PR_TEST_ROOT/project"
start=$(date +%s.%N)
output=$($PR_BIN source list -o text 2>&1)
end=$(date +%s.%N)
duration=$(echo "$end - $start" | bc)

if [ $? -eq 0 ] && echo "$output" | grep -q "detached-foo"; then
    status="pass"
else
    status="fail"
fi

jq --arg s "5" --arg n "source-list" --arg st "$status" --arg o "$output" --arg d "$duration" \
   '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
   "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"

echo "Source list: $status"
echo "$output"

In [ ]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
export XDG_CONFIG_HOME="$PR_TEST_ROOT/xdg"
export REPO_ROOT=$(cd /home/wherka/workspace/opensource/prompt-registry && pwd)
export PR_BIN="node $REPO_ROOT/lib/dist/cli/main.js"
EOF

# 5.3 Remove source
cd "$PR_TEST_ROOT/project"
start=$(date +%s.%N)
output=$($PR_BIN source remove detached-foo -o json 2>&1)
end=$(date +%s.%N)
duration=$(echo "$end - $start" | bc)

if echo "$output" | jq -e '.status=="ok"' > /dev/null; then
    status="pass"
else
    status="fail"
fi

jq --arg s "5" --arg n "source-remove" --arg st "$status" --arg o "$output" --arg d "$duration" \
   '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
   "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"

echo "Source remove: $status"

## 6. End-to-End Profile Activation (Synthetic Bundle)

In [ ]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
EOF

# 6.1 Build synthetic bundle
bundle_dir="$PR_TEST_ROOT/bundles/local-foo"
mkdir -p "$bundle_dir/prompts"

cat > "$bundle_dir/deployment-manifest.yml" <<'EOF'
id: local-foo
version: 1.0.0
name: Local Foo
EOF

echo "# A prompt" > "$bundle_dir/prompts/a.md"

echo "✓ Synthetic bundle created at $bundle_dir"

In [ ]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
EOF

# 6.2 Build local hub config
hub_dir="$PR_TEST_ROOT/local-hub"
mkdir -p "$hub_dir"

cat > "$hub_dir/hub-config.yml" <<EOF
version: 1.0.0
metadata:
  name: Local Test Hub
  description: synthetic hub for the manual test plan
  maintainer: tester
  updatedAt: '2026-04-26T00:00:00Z'
sources:
  - id: local-foo-src
    name: Local Foo Source
    type: local
    url: $PR_TEST_ROOT/bundles/local-foo
    enabled: true
    priority: 0
    hubId: local-test-hub
profiles:
  - id: backend
    name: Backend Developer
    bundles:
      - id: local-foo
        version: 1.0.0
        source: local-foo-src
        required: true
EOF

echo "✓ Local hub config created at $hub_dir"

In [ ]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
export XDG_CONFIG_HOME="$PR_TEST_ROOT/xdg"
export REPO_ROOT=$(cd /home/wherka/workspace/opensource/prompt-registry && pwd)
export PR_BIN="node $REPO_ROOT/lib/dist/cli/main.js"
EOF

# 6.3 Import hub
cd "$PR_TEST_ROOT/project"
start=$(date +%s.%N)
output=$($PR_BIN hub add --type local --location "$PR_TEST_ROOT/local-hub" -o json 2>&1)
end=$(date +%s.%N)
duration=$(echo "$end - $start" | bc)

if echo "$output" | jq -e '.status=="ok"' > /dev/null; then
    status="pass"
else
    status="fail"
fi

jq --arg s "6" --arg n "hub-import" --arg st "$status" --arg o "$output" --arg d "$duration" \
   '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
   "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"

echo "Hub import: $status"
echo "$output"

In [ ]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
export XDG_CONFIG_HOME="$PR_TEST_ROOT/xdg"
export REPO_ROOT=$(cd /home/wherka/workspace/opensource/prompt-registry && pwd)
export PR_BIN="node $REPO_ROOT/lib/dist/cli/main.js"
EOF

# 6.4 Activate hub
cd "$PR_TEST_ROOT/project"
start=$(date +%s.%N)
output=$($PR_BIN hub use local-test-hub -o json 2>&1)
end=$(date +%s.%N)
duration=$(echo "$end - $start" | bc)

if echo "$output" | jq -e '.status=="ok" and .data.activeId=="local-test-hub"' > /dev/null; then
    status="pass"
else
    status="fail"
fi

jq --arg s "6" --arg n "hub-activate" --arg st "$status" --arg o "$output" --arg d "$duration" \
   '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
   "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"

echo "Hub activate: $status"
echo "$output"

In [ ]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
export XDG_CONFIG_HOME="$PR_TEST_ROOT/xdg"
export REPO_ROOT=$(cd /home/wherka/workspace/opensource/prompt-registry && pwd)
export PR_BIN="node $REPO_ROOT/lib/dist/cli/main.js"
EOF

# 6.5 Show profile
cd "$PR_TEST_ROOT/project"
start=$(date +%s.%N)
output=$($PR_BIN profile show backend -o json 2>&1)
end=$(date +%s.%N)
duration=$(echo "$end - $start" | bc)

if echo "$output" | jq -e '.status=="ok" and .data.profile.id=="backend"' > /dev/null; then
    status="pass"
else
    status="fail"
fi

jq --arg s "6" --arg n "profile-show" --arg st "$status" --arg o "$output" --arg d "$duration" \
   '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
   "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"

echo "Profile show: $status"
echo "$output"

## 6.6 Profile Activation (SKIP - requires target add)

In [ ]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
EOF

jq --arg s "6" --arg n "profile-activate-skip" --arg st "skip" --arg o "Skipped - requires target add (defineCommand limitation)" --arg d "0" \
   '.sections[$s] |= if .tests then .tests += [{name: $n, status: $st, output: $o, duration: $d}] else {tests: [{name: $n, status: $st, output: $o, duration: $d}]} end' \
   "$PR_TEST_ROOT/results.json" > "$PR_TEST_ROOT/results.tmp" && mv "$PR_TEST_ROOT/results.tmp" "$PR_TEST_ROOT/results.json"

echo "Profile activation skipped - requires target add first (defineCommand limitation)"

## Generate Test Report

In [ ]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
EOF

results_file="$PR_TEST_ROOT/results.json"
report_file="$PR_TEST_ROOT/test-report.html"

# Calculate statistics
total=$(jq '[.sections[].tests[]] | length' "$results_file")
passed=$(jq '[.sections[].tests[] | select(.status=="pass")] | length' "$results_file")
failed=$(jq '[.sections[].tests[] | select(.status=="fail")] | length' "$results_file")
skipped=$(jq '[.sections[].tests[] | select(.status=="skip")] | length' "$results_file")

if [ "$total" -gt 0 ]; then
    pass_rate=$(echo "scale=1; $passed * 100 / $total" | bc)
else
    pass_rate="0"
fi

# Generate HTML report
cat > "$report_file" <<HTML
<!DOCTYPE html>
<html>
<head>
    <title>Manual Test Plan Report</title>
    <style>
        body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; margin: 40px; }
        .summary { background: #f5f5f5; padding: 20px; border-radius: 8px; margin-bottom: 30px; }
        .section { margin-bottom: 30px; border: 1px solid #ddd; border-radius: 8px; overflow: hidden; }
        .section-header { background: #333; color: white; padding: 15px; }
        .test { padding: 15px; border-bottom: 1px solid #eee; }
        .test:last-child { border-bottom: none; }
        .pass { color: green; font-weight: bold; }
        .fail { color: red; font-weight: bold; }
        .skip { color: orange; font-weight: bold; }
        pre { background: #f5f5f5; padding: 10px; border-radius: 4px; overflow-x: auto; max-height: 200px; }
    </style>
</head>
<body>
    <div class="summary">
        <h1>Manual Test Plan Report</h1>
        <p>Generated: $(date)</p>
        <h2>Summary</h2>
        <p>Total tests: $total</p>
        <p class="pass">Passed: $passed</p>
        <p class="fail">Failed: $failed</p>
        <p class="skip">Skipped: $skipped</p>
        <p>Pass rate: ${pass_rate}%</p>
    </div>
HTML

# Add sections
jq -r '.sections | to_entries[] | "\(.key): \(.value.name)"' "$results_file" | while read -r line; do
    section_id=$(echo "$line" | cut -d: -f1)
    section_name=$(echo "$line" | cut -d: -f2-)
    
    cat >> "$report_file" <<HTML
    <div class="section">
        <div class="section-header"><h2>Section ${section_id}: ${section_name}</h2></div>
HTML
    
    jq -r ".sections[\"$section_id\"].tests[] | \"<div class=\"test\"><h3 class=\"\(.status)\">\(.status | ascii_upcase): \(.name)</h3>\" + (if .duration then "<p>Duration: \(.duration)s</p>" else "" end) + (if .output then "<p>Output:</p><pre>\(.output[:500])</pre>" else "" end) + "</div>\"" "$results_file" >> "$report_file"
    
    echo "    </div>" >> "$report_file"
done

echo "</body></html>" >> "$report_file"

echo "✓ Report generated: $report_file"
echo ""
echo "=== Test Summary ==="
jq -r '.sections | to_entries[] | "Section \(.key):" + ([.value.tests[] | "  \(.status | ascii_upcase): \(.name)"] | join("\n"))' "$results_file"

## Teardown

In [ ]:
%%bash

source /dev/stdin <<'EOF'
export PR_TEST_ROOT=$HOME/.test-prompt-registry
EOF

# Clean up test directories (optional - comment out to preserve state)
# rm -rf "$PR_TEST_ROOT"
# echo "Cleaned up: $PR_TEST_ROOT"
echo "Teardown skipped - preserving test state for inspection"